# Clase 28: Ética, sesgo y privacidad en datos

En esta clase exploraremos cómo detectar sesgo en un dataset real y por qué la ética es una parte fundamental del trabajo del científico de datos.

> **Idea central:** Un modelo técnicamente correcto puede ser socialmente injusto.

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('seaborn-v0_8')
print('Librerías cargadas correctamente')

## 1. Cargar y explorar el dataset

Vamos a trabajar con un dataset de estudiantes. Antes de cualquier análisis, siempre preguntamos: **¿quién está en estos datos y quién no?**

In [ ]:
# Crear dataset de ejemplo (en clase real se usaría el archivo CSV)
np.random.seed(42)
n = 300

# Simulamos un dataset con sesgo: el curso C tiene peores condiciones (menos recursos)
cursos = np.random.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
edades = np.random.randint(18, 35, size=n)

nota_mate = np.where(
    cursos == 'A', np.random.normal(7.5, 1.2, n),
    np.where(cursos == 'B', np.random.normal(7.0, 1.3, n),
             np.random.normal(5.8, 1.5, n))
).clip(1, 10)

nota_lengua = np.where(
    cursos == 'A', np.random.normal(7.2, 1.1, n),
    np.where(cursos == 'B', np.random.normal(6.8, 1.2, n),
             np.random.normal(5.5, 1.4, n))
).clip(1, 10)

asistencia = np.where(
    cursos == 'C', np.random.uniform(0.55, 0.85, n),
    np.random.uniform(0.70, 0.99, n)
)

# La aprobación depende de las notas y asistencia
prob_aprobar = (nota_mate * 0.4 + nota_lengua * 0.4 + asistencia * 10 * 0.2) / 10
aprobado = (np.random.uniform(0, 1, n) < prob_aprobar).astype(int)

df = pd.DataFrame({
    'curso': cursos,
    'edad': edades,
    'nota_matematicas': nota_mate.round(1),
    'nota_lengua': nota_lengua.round(1),
    'asistencia': asistencia.round(2),
    'aprobado': aprobado
})

print(f'Dataset: {df.shape[0]} estudiantes, {df.shape[1]} columnas')
df.head()

## 2. Análisis de distribución por grupos

**Pregunta clave:** ¿Están todos los grupos igualmente representados?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de estudiantes por curso
conteo = df['curso'].value_counts()
axes[0].bar(conteo.index, conteo.values, color=['#4C72B0', '#55A868', '#C44E52'])
axes[0].set_title('Cantidad de estudiantes por curso')
axes[0].set_xlabel('Curso')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 1, str(v), ha='center')

# Tasa de aprobación por curso
tasas = df.groupby('curso')['aprobado'].mean()
axes[1].bar(tasas.index, tasas.values, color=['#4C72B0', '#55A868', '#C44E52'])
axes[1].set_title('Tasa de aprobación por curso')
axes[1].set_xlabel('Curso')
axes[1].set_ylabel('Tasa de aprobación')
axes[1].set_ylim(0, 1)
for i, v in enumerate(tasas.values):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center')

plt.tight_layout()
plt.show()

print('\n=== Tasa de aprobación por curso ===')
print(tasas.round(3))

## 3. Calcular el impacto dispar

La **regla del 80%** dice que si el grupo menos favorecido recibe resultados positivos con menos del 80% de la tasa del grupo más favorecido, existe impacto dispar.

In [ ]:
tasas = df.groupby('curso')['aprobado'].mean()

tasa_max = tasas.max()
tasa_min = tasas.min()
curso_max = tasas.idxmax()
curso_min = tasas.idxmin()

razon = tasa_min / tasa_max

print('=== Análisis de Impacto Dispar ===')
print(f'Curso con mayor tasa de aprobación: {curso_max} ({tasa_max:.1%})')
print(f'Curso con menor tasa de aprobación: {curso_min} ({tasa_min:.1%})')
print(f'Razón de impacto dispar: {razon:.2f}')
print()

if razon < 0.8:
    print('⚠️  ALERTA: Posible impacto dispar (razón < 0.80)')
    print(f'   El curso {curso_min} aprueba a {razon:.1%} de la tasa del curso {curso_max}')
else:
    print('✓  No se detecta impacto dispar significativo (razón >= 0.80)')

print()
print('IMPORTANTE: El impacto dispar no siempre significa discriminación injusta.')
print('Hay que investigar las causas — pueden ser estructurales o contextuales.')

## 4. Entrenar un modelo y evaluarlo por subgrupos

Un error común: evaluar solo la precisión global. Un modelo puede tener 90% de precisión general y al mismo tiempo fallar sistemáticamente con un grupo específico.

In [ ]:
features = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
X = df[features]
y = df['aprobado']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Guardamos los grupos del test set
grupos_test = df.loc[X_test.index, 'curso']

# Entrenar modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

# Precisión global
precision_global = accuracy_score(y_test, predicciones)
print(f'Precisión global del modelo: {precision_global:.2%}')
print()

# Precisión por curso
print('=== Precisión del modelo por curso ===')
resultados = []
for curso in sorted(df['curso'].unique()):
    mask = grupos_test == curso
    if mask.sum() > 0:
        acc = accuracy_score(y_test[mask], predicciones[mask])
        n = mask.sum()
        resultados.append({'Curso': curso, 'Precisión': acc, 'N estudiantes': n})
        print(f'  Curso {curso}: {acc:.2%} (n={n})')

print()
print('¿El modelo tiene el mismo rendimiento para todos los grupos?')

## 5. Importancia de variables — ¿qué está usando el modelo?

Si un modelo usa variables que son proxies de características protegidas (como el curso, que puede estar correlacionado con la clase socioeconómica), puede perpetuar desigualdades.

In [ ]:
importancias = pd.Series(
    modelo.feature_importances_,
    index=features
).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importancias.plot(kind='barh', color='#4C72B0')
plt.title('Importancia de cada variable en el modelo')
plt.xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

print('Importancia de variables:')
for var, imp in importancias.sort_values(ascending=False).items():
    print(f'  {var}: {imp:.3f}')

print()
print('REFLEXIÓN: ¿La edad debería influir en si un estudiante aprueba?')
print('Si influye mucho, ¿podría eso ser una forma indirecta de discriminación?')

## 6. Privacidad: demostración de anonimización

In [ ]:
import hashlib

# Supongamos que el dataset original tenía nombres
nombres_ficticios = ['Ana G.', 'Juan P.', 'María L.', 'Carlos R.', 'Sofía M.']
df_con_nombres = df.head(5).copy()
df_con_nombres.insert(0, 'nombre', nombres_ficticios)

print('Dataset ANTES de anonimizar:')
print(df_con_nombres[['nombre', 'curso', 'edad', 'aprobado']])

print()

# Técnica 1: Eliminar el nombre
df_anonimo = df_con_nombres.drop(columns=['nombre'])
print('Dataset DESPUÉS de eliminar nombre:')
print(df_anonimo[['curso', 'edad', 'aprobado']])

print()
print('⚠️  ADVERTENCIA: Solo eliminar el nombre no siempre es suficiente.')
print('La combinación de curso + edad + otras variables puede ser suficiente')
print('para re-identificar a una persona en un dataset pequeño.')

## 7. Concepto de SHAP Values

SHAP (SHapley Additive exPlanations) nos permite entender **por qué** el modelo tomó una decisión específica para un estudiante en particular.

In [ ]:
# Simulamos la explicación SHAP para un estudiante
# (En producción usarías: pip install shap; import shap)

estudiante_ejemplo = X_test.iloc[0]
prediccion = modelo.predict([estudiante_ejemplo])[0]
prob = modelo.predict_proba([estudiante_ejemplo])[0][1]

print('=== Explicación de predicción para un estudiante ===')
print(f'Datos del estudiante:')
for col, val in estudiante_ejemplo.items():
    print(f'  {col}: {val}')

print(f'\nPredicción del modelo: {"APROBADO" if prediccion == 1 else "REPROBADO"}')
print(f'Probabilidad de aprobar: {prob:.1%}')

print()
print('Con SHAP podríamos ver algo como:')
print('  nota_matematicas = 8.2  →  +0.23 (aumenta probabilidad de aprobar)')
print('  asistencia = 0.91       →  +0.18')
print('  nota_lengua = 6.5       →  +0.05')
print('  edad = 28               →  -0.03 (disminuye levemente)')
print()
print('Esto permite EXPLICAR la decisión a cualquier persona afectada.')
print('La transparencia es un derecho, no un extra.')

## 8. Discusión final: Principios de IA Responsable

| Principio | Pregunta que debes hacerte |
|-----------|---------------------------|
| **Equidad** | ¿El modelo funciona igual de bien para todos los grupos? |
| **Transparencia** | ¿Puedo explicar por qué el modelo tomó esta decisión? |
| **Privacidad** | ¿Estoy usando solo los datos necesarios? ¿Con consentimiento? |
| **Responsabilidad** | ¿Hay un humano responsable de las decisiones finales? |
| **Seguridad** | ¿Podría alguien manipular el modelo para obtener resultados injustos? |

### Checklist mínimo antes de desplegar un modelo:

- [ ] Analicé la distribución del dataset por grupos
- [ ] Evalué el rendimiento por subgrupos, no solo global
- [ ] Calculé métricas de equidad relevantes
- [ ] Los datos personales están anonimizados
- [ ] Tengo una forma de explicar las predicciones
- [ ] Existe un proceso humano de revisión para casos límite

**La ética en datos no es opcional — es parte del trabajo profesional.**